In [ ]:
# import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import polars as pl
from datetime import datetime, timedelta

from matplotlib.dates import MonthLocator



import matplotlib.dates as mdates 
import matplotlib.transforms as mtransforms
import scienceplots
import seaborn as sns
cmp = sns.color_palette('Set2')

import oracledb
import pandas as pd


from matplotlib.dates import MonthLocator
from matplotlib.dates import DateFormatter


In [ ]:
run = 1

if run:
    def query_full(): 
    # specify your OracleSQL query (or multiple ones) 
        q1 = f'''
        INSERT QUERY FOR ALL POSITIVE RT-PCR TESTS BETWEEN 01-09-2020 AND 31-12-2021
        '''
        return([q1])

    def retrieve_data(query):
        # extract data from DB for all pids
        oracledb.init_oracle_client()
        with oracledb.connect(user='', dsn='') as con:
            with con.cursor() as cur:
                out = [np.asarray(cur.execute(qq).fetchall()) for qq in query()]
                cols = cur.description
        return(out, cols)



    data_full, columns_full = retrieve_data(query_full)
    cols_full = [col[0] for col in columns_full]

    df_full = pd.DataFrame(data_full[0], columns=cols_full)


In [ ]:
df_episodes = pl.from_pandas(df_full.copy())
def drop_episode_ids(df_person):
    testdates = df_person.select('PRDATE').to_numpy().flatten()
    episode_numbers = [((t - testdates[0]).astype('timedelta64[D]')).astype(int) // 60 for t in testdates]
    
    df_person = df_person.with_columns(pl.Series(name = 'episode', values = episode_numbers))
    keep_ids = df_person.groupby('episode').first().select('ROWID').to_numpy().flatten().tolist()

    reject_ids = df_person.filter(~pl.col('ROWID').is_in(keep_ids)).select('ROWID').to_numpy().flatten()
    return list(reject_ids)
for i in tqdm(df_full.PERSON_ID.unique()):
    df_person = df_episodes.filter(pl.col('PERSON_ID') == i)
    if len(df_person) > 1:
        reject_ids = drop_episode_ids(df_person)
        df_episodes = df_episodes.filter(~pl.col('ROWID').is_in((reject_ids)))
df_episodes.write_csv('./df_episodes_new.csv')      

In [ ]:
seqs_by_day = np.genfromtxt('./count_sequences_by_day.csv').astype(int)
cases_by_day = df_episodes.group_by(pl.col('PRDATE_ADJUSTED')).len().rename({'len' : 'count'}).sort('PRDATE_ADJUSTED').to_numpy()[:, 1]

In [ ]:
df_episodes_seqs = pl.read_csv('./df_full_sequences_episodes.csv')

In [ ]:
df_episodes.group_by(pl.col('PRDATE_ADJUSTED')).len().rename({'len' : 'count'}).sort('PRDATE_ADJUSTED').filter(pl.col('PRDATE_ADJUSTED')>=
                                                                                      pl.date(2021, 1, 1))

In [ ]:
from datetime import timedelta

df_seqs_dates = df_episodes_seqs.select(pl.col('PERSON_ID', 'DATESAMPLING'))

df_seq_episodes = df_episodes.join(df_seqs_dates,
                 on = 'PERSON_ID',
                 how = 'left').filter(pl.col('DATESAMPLING').str.to_date() <= 
                                      pl.col('PRDATE_ADJUSTED') + timedelta(60)).unique(['PERSON_ID', 'PRDATE_ADJUSTED'])
df_person_dates = df_seq_episodes.select(pl.col('PERSON_ID',
                                                'DATESAMPLING')).with_columns(pl.col('DATESAMPLING').str.to_date())

df_person_dates_filter = df_person_dates.filter(pl.col('DATESAMPLING') <
                              pl.date(2022, 1, 1)).unique().sort('DATESAMPLING').group_by('DATESAMPLING').len().rename({'len' : 'count'})
df_person_dates_filter = df_person_dates_filter.filter(pl.col('DATESAMPLING').is_between(pl.date(2020, 9, 1), pl.date(2021, 12, 31))).to_numpy()


df_seqs_dates.filter(pl.col('DATESAMPLING')=='2021-06-27').unique()
boop = df_person_dates.filter(pl.col('DATESAMPLING') <
                              pl.date(2022, 1, 1)).sort('DATESAMPLING').group_by('DATESAMPLING').len().rename({'len' : 'count'})

df_person_dates.filter(pl.col('DATESAMPLING') <
                              pl.date(2022, 1, 1)).filter(pl.col('DATESAMPLING') == pl.date(2021, 6, 27)).unique()

In [ ]:
ssi_data = pd.read_csv('../../covid_SSI_epi_data/08_bekraeftede_tilfaelde_pr_dag_pr_regions.csv', sep = ';', encoding='unicode_escape')
ssi_data = ssi_data.loc[ssi_data['Dato'].between('2020-08-11', '2022-04-11')]
ssi_data['Dato'].values
ssi_data_pl = pl.from_pandas(ssi_data.loc[ssi_data['Dato'].between('2020-08-11', '2021-12-31')])
cases = ssi_data_pl.group_by('Dato').sum().rename({'Bekræftede tilfælde i alt' : 
                                          'Cases'}).select(pl.col('Dato', 'Cases')).sort(pl.col('Dato'))

In [ ]:
df_episodes.group_by(pl.col('PRDATE_ADJUSTED')).len().rename({'len' : 'count'}).sort('PRDATE_ADJUSTED')[320, :]
cases_new = cases.filter(pl.col('Dato').str.to_date().is_between(pl.date(2020, 9, 1), pl.date(2021, 12, 31)))

In [ ]:
date_strs = [d.date() for d in pd.date_range(start='9/1/2020', end ='31/12/2021', freq='D')]
cases_dict = {'date' : date_strs, 'sequencing_proportion' : np.array(all_seqs_array / all_cases_array)}
pd.DataFrame(cases_dict).to_csv('../seq_proportion_2020_2021.csv')

In [ ]:
import pygam
from pygam import s, f

all_seqs_array = beep[:, 1]
all_cases_array = cases_new.to_numpy()[:, 1]
X = np.arange(len(all_cases_array))
y = np.array(all_seqs_array / all_cases_array)
gam = pygam.LinearGAM(s(0)).fit(X, y)

gam_line = np.zeros((100))
for i, term in enumerate(gam.terms):

    if term.isintercept:
        continue
    XX = gam.generate_X_grid(term=i)
    pdep, confi = gam.partial_dependence(term=i, X=XX, width=0.95)
    intercept = gam.coef_[-1]
gam_line[:] = pdep+intercept




fig = plt.figure(figsize = (15, 10))

ax = fig.gca()
plt.plot(XX, gam_line, lw = 3)
plt.plot(X, y, lw = 3, color = 'grey', alpha = 0.5)
date_labs = [d.date().strftime('%b\n%Y') for d in pd.date_range(start='9/1/2020', end ='31/12/2021', freq='D')]
ax.xaxis.set_ticks(np.arange(487).astype(int), date_labs)
    
ax.xaxis.set_major_locator(MonthLocator(np.arange(12).astype(int) + 1))
plt.ylabel('Proportion of Cases')
plt.xlim([0, 487])
plt.ylim([0, 1])
plt.xlabel('Date')
plt.title('Daily Proportion of Cases with an associated sequence in Denmark in 2021')
plt.grid(alpha = 0.5)


In [ ]:
# Read in IAR estimates - generated externally
dk_iar = pd.read_csv('../../denmark_rt_iar.csv')
dk_iar = dk_iar.loc[dk_iar['date']>= '2020-09-01']
dk_iar_array = dk_iar.iar.values
dk_iar_array_lq = dk_iar['iar_0.05'].values
dk_iar_array_uq = dk_iar['iar_0.95'].values

gam_line_iar = np.zeros((100))
for i, term in enumerate(gam.terms):

    if term.isintercept:
        continue
    XX = gam.generate_X_grid(term=i)
    pdep, confi = gam.partial_dependence(term=i, X=XX, width=0.95)
    intercept = gam.coef_[-1]
gam_line_iar[:] = pdep+intercept

In [ ]:
plt.style.use('science')

def get_sum_over_days(array, days = 11):
    summation = np.zeros(len(array))
    for i in range(len(array)):
        if i <= days:
            summation[i] = np.sum(array[:(i+1)])
        elif i == len(array)-1:
            summation[i] = np.sum(array[(i - days):])
        else:
            summation[i] = np.sum(array[(i - days):(i+1)])
            
    return summation

prop_1 = get_sum_over_days((all_cases_array / dk_iar_array))
prop_1_lq = get_sum_over_days((all_cases_array / dk_iar_array_uq))
prop_1_uq = get_sum_over_days((all_cases_array / dk_iar_array_lq))
prop_0 = get_sum_over_days(all_seqs_array)
fig = plt.figure(figsize = (12, 6))
ax = fig.gca()
plt.grid(alpha = 0.5)
plt.plot(X, prop_0 / prop_1, linewidth = 3, color = 'tab:blue', label = 'Proportion')
plt.plot(X, dk_iar_array, alpha = 0.5, color = 'tab:purple', linewidth = 3, label = 'IAR (daily, modelled)', ls = '--')
plt.fill_between(X, dk_iar_array_lq, dk_iar_array_uq, 
                 alpha = 0.1, color = 'tab:purple')
plt.fill_between(X, prop_0 / prop_1_uq, prop_0 / prop_1_lq, 
                 alpha = 0.25, color = 'tab:blue')

plt.plot(XX, gam_line_iar, alpha = 0.5, color = 'tab:orange', linewidth = 3, label= 'Sequencing Proportion (daily)', ls = '--')

date_labs = [d.date().strftime('%b\n%Y') for d in pd.date_range(start='9/30/2020', end ='1/1/2022', freq='D')]
ax.xaxis.set_ticks(np.arange(459).astype(int), date_labs)
    
ax.xaxis.set_major_locator(MonthLocator())
plt.ylabel('Proportion', fontsize = 14)
# plt.xlim([0, 365])
plt.ylim([0, 1])
plt.xlabel('Date', fontsize = 14)
plt.title('Estimated Daily Proportion of Infections Sequenced in the Previous 11 Days in 2021', fontsize = 16)
plt.title('A', loc='left', fontsize = 16)
plt.legend()

plt.xlim([X[0], X[-1]])
plt.xticks(fontsize = 12)
plt.yticks(fontsize = 12)

# plt.savefig('../Figures/sequencing_proportion_2020_21.pdf')